In [1]:
import json
import pandas as pd
import numpy as np

In [2]:
df = pd.read_excel("../india_weather_rainfall_data.xlsx")
df

,date_of_record,month,season,station_name,state,district,avg_temp,min_temp,max_temp,wind_speed,air_pressure,elevation,latitude,longitude,rainfall
0,2021-01-02,January,Winter,Gulmarg,JK,Baramulla,-2.2,-6.6,-0.8,2.2,1020.0,2652,34.0500,74.4000,0.1
1,2021-01-03,January,Winter,Gulmarg,JK,Baramulla,-3.6,-4.6,-1.8,3.7,1019.5,2652,34.0500,74.4000,4.4
2,2021-01-04,January,Winter,Gulmarg,JK,Baramulla,-3.0,-4.5,-1.1,2.1,1022.0,2652,34.0500,74.4000,2.3
3,2021-01-05,January,Winter,Gulmarg,JK,Baramulla,-3.3,-5.1,-1.2,2.8,1015.6,2652,34.0500,74.4000,35.0
4,2021-01-06,January,Winter,Gulmarg,JK,Baramulla,-3.9,-8.3,-1.0,3.4,1015.3,2652,34.0500,74.4000,25.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
970334,2025-02-06,February,Winter,Vizagapatam / Gajuwaka,AP,Visakhapatnam,27.3,22.0,34.0,7.6,1011.4,5,17.7167,83.2333,0.0
970335,2025-02-07,February,Winter,Vizagapatam / Gajuwaka,AP,Visakhapatnam,26.7,23.0,30.0,7.7,1013.1,5,17.7167,83.2333,0.3
970336,2025-02-08,February,Winter,Vizagapatam / Gajuwaka,AP,Visakhapatnam,26.5,23.0,30.0,8.7,1013.0,5,17.7167,83.2333,0.0
970337,2025-02-09,February,Winter,Vizagapatam / Gajuwaka,AP,Visakhapatnam,26.5,23.0,30.0,8.5,1012.8,5,17.7167,83.2333,0.0


In [31]:
df = df.replace({np.nan: None})

In [32]:
for col in df.select_dtypes(include=['datetime64[ns]']):
    df[col] = df[col].astype(str)

In [33]:
missing_values = {}
null_count = df.isnull().sum()
for n, i in enumerate(null_count):
    missing_values[df.columns[n]] = 100 - (df.shape[0] - i)/df.shape[0]*100

missing_values

{'date_of_record': 0.0,
 'month': 0.0,
 'season': 0.0,
 'station_name': 0.0,
 'state': 0.0,
 'district': 0.0,
 'avg_temp': 0.0,
 'min_temp': 4.523985947179284,
 'max_temp': 11.3978722899935,
 'wind_speed': 28.283311296361376,
 'air_pressure': 31.397686787813328,
 'elevation': 0.0,
 'latitude': 0.0,
 'longitude': 0.0,
 'rainfall': 26.542682505804677}

In [34]:
def district_groupwise_data(target):
    district_avg_temp_dict = df.groupby(['state','district'])[target].mean().reset_index().to_dict(orient='records')
    district_avg_temp = {}

    for i in district_avg_temp_dict:
        if i['state'] not in district_avg_temp:
            district_avg_temp[i['state']] = {}

        district_avg_temp[i['state']][i['district']] = i[target]

    return district_avg_temp

In [35]:
def station_groupwise_data(target):
    station_avg_temp_dict = df.groupby(['state','district','station_name'])[target].mean().reset_index().to_dict(orient='records')
    station_avg_temp = {}

    for i in station_avg_temp_dict:
        if i['state'] not in station_avg_temp:
            station_avg_temp[i['state']] = {}

        if i['district'] not in station_avg_temp[i['state']]:
            station_avg_temp[i['state']][i['district']] = {}

        station_avg_temp[i['state']][i['district']][i['station_name']] = i[target]

    return station_avg_temp

In [36]:
def season_groupwise_data(target):
    season_avg_temp_dict = df.groupby(['state','district','station_name', 'season'])[target].mean().reset_index().to_dict(orient='records')
    season_avg_temp = {}

    for i in season_avg_temp_dict:
        if i['state'] not in season_avg_temp:
            season_avg_temp[i['state']] = {}

        if i['district'] not in season_avg_temp[i['state']]:
            season_avg_temp[i['state']][i['district']] = {}

        if i['station_name'] not in season_avg_temp[i['state']][i['district']]:
                season_avg_temp[i['state']][i['district']][i['station_name']] = {}

        season_avg_temp[i['state']][i['district']][i['station_name']][i['season']] = i[target]

    return season_avg_temp

In [39]:
data = {
    "shape":df.shape,
    "columns": df.columns.tolist(),
    "state": df['state'].unique().tolist(),
    "district": df.groupby('state')['district'].unique().apply(list).to_dict(),
    "station": df.groupby('district')['station_name'].unique().apply(list).to_dict(),
    "season": df.groupby('station_name')['season'].unique().apply(list).to_dict(),
    "missing_values": missing_values,
    "null_count": null_count.to_dict(),
    "state_avg_temp": df.groupby('state')['avg_temp'].mean().to_dict(),
    "district_avg_temp": district_groupwise_data('avg_temp'),
    "station_avg_temp": station_groupwise_data('avg_temp'),
    "season_avg_temp": season_groupwise_data('avg_temp'),
    "state_rainfall": df.groupby('state')['rainfall'].mean().to_dict(),
    "district_rainfall": district_groupwise_data('rainfall'),
    "station_rainfall": station_groupwise_data('rainfall'),
    "season_rainfall": season_groupwise_data('rainfall'),
    "df_date_of_record": df['date_of_record'].astype(str).tolist(),
    "dataset": df.to_dict(orient='records'),
}

In [40]:
with open('data.json', 'w') as f:
    json.dump(data, f, indent=4)

print("Data dumped successfully!")

Data dumped successfully!
